# Chapter 2: The Model Validation Framework

A fixed predictive model can have a low average error while concentrating larger errors in part of its population. Chapter 2 frames validation as a search for such weaknesses under a declared test space: candidate tests in an Inner Array, evaluation conditions in an Outer Array, measured results compared on a Pareto frontier and selected findings confirmed on reserved observations. This notebook follows one test from the synthetic AML case through that workflow.

The predictor is a saved logistic model of a rare AML outcome and remains fixed throughout. A saved depth-two XGBoost model of the predictor's Brier loss supplies the representation in which customer regions are formed. The notebook reads the recorded search history, recomputes the empirical frontier and the primary selection, replays the frozen region assignment on confirmation observations, recomputes the confirmation measurement and checks a reported value against the evidence store. No model is fitted and no language model is called.

**Evidence.** The records come from the historical medoid-loop experiment in `book/evidence/aml_medoid_loop_leaf_output_20260921/`. It uses 40,000 discovery observations, a fixed 3,000-row medoid pool and 200,000 confirmation observations drawn with seed 62491. The population is synthetic, and the measurements concern the frozen predictor in that population. Chapter 3's corrected walkthrough uses a different geometry and a different confirmation sample; its numbers should not be combined with these.

**Run order.** Run the cells from top to bottom. Later cells use objects created earlier, and each assertion checks a recomputed quantity against the saved record. A passing assertion shows agreement with the record; it does not validate the modeling assumptions.

## Setup and source checks

Use a Python kernel with the dependencies listed in the README. The setup cell locates the data root, which holds the evidence files and helper modules. It uses `MODEL_VALIDATION_ROOT` when that variable is set. Otherwise it searches for a repository checkout containing the current directory and then for the copy installed with the `aiv` package. It prints the Python version and which source it used.

`load()` reads the protocol, the confirmation results and the frozen finalists. Before returning, it checks the SHA-256 digests that link these files. The results record the digest of the frozen-finalists file, and the finalists record the digests of the protocol, predictor, auxiliary models and admission record. The run's input manifest records the digests of the discovery artifacts it used. A failed assertion here means a file differs from the recorded run and should be resolved before the replay is interpreted. `RUN` is the evidence directory; `frontier`, `confirmation`, `measure` and `assignments` are the helper functions used below.

In [1]:
from pathlib import Path
import os
import platform
import sys
import importlib.util

# Locate the data root: the directory whose relative layout the evidence records use.
MARKER = "book/notebooks/regional_case/companion.py"

def find_data_root():
    configured = os.environ.get("MODEL_VALIDATION_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        if not (root / MARKER).is_file():
            raise FileNotFoundError(f"MODEL_VALIDATION_ROOT must contain {MARKER}")
        return root, "MODEL_VALIDATION_ROOT"
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / MARKER).is_file():
            return path, "repository checkout"
    try:
        from aiv.companions import locate
    except ImportError:
        raise FileNotFoundError(
            "Companion data not found. Install the aiv package, open the notebook "
            "inside the repository checkout, or set MODEL_VALIDATION_ROOT.") from None
    return locate()

ROOT, DATA_SOURCE = find_data_root()
HELPERS = ROOT / "book/notebooks/regional_case"
for path in (ROOT, ROOT / "book", HELPERS):
    value = str(path)
    if value in sys.path:
        sys.path.remove(value)
    sys.path.insert(0, value)

# Load current helper files even if this kernel cached an older notebook revision.
def load_helper_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    try:
        spec.loader.exec_module(module)
    except Exception:
        sys.modules.pop(name, None)
        raise
    return module

load_helper_module("companion", HELPERS / "companion.py")
load_helper_module("ch02_helpers", HELPERS / "ch02_helpers.py")
load_helper_module("weighted_leaf_geometry", ROOT / "book/weighted_leaf_geometry.py")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
print("Python:", platform.python_version())
print("Data root found from:", DATA_SOURCE)
from companion import RUN, read, load, frontier, confirmation, measure, assignments
pd.set_option("display.float_format", "{:.6f}".format)
protocol, results, frozen = load()
print("Evidence:", RUN.relative_to(ROOT).as_posix())
print("Mode: saved-evidence replay; no fitting or live LLM calls")


Python: 3.12.3
Data root found from: repository checkout
Evidence: book/evidence/aml_medoid_loop_leaf_output_20260921
Mode: saved-evidence replay; no fitting or live LLM calls


## Identify the test

A validation test for the frozen predictor must say which observations it examines before any loss is measured. In this study a test is a customer region, and the region is produced by an executable rule. The rule has two parts. Four medoids, each an observed discovery row, partition the observations by nearest medoid. A declared selection rule then picks one of the four cells as the region under test. One Inner Array row is therefore a complete four-medoid candidate together with the cell it selects (Section 2.5).

Distances are computed in the representation supplied by the fixed auxiliary error model. Each observation is described by the 120 per-tree leaf outputs of the saved depth-two XGBoost model of Brier loss, and Euclidean distance compares those vectors without standardization. This historical loop uses per-tree outputs; Chapter 3's corrected walkthrough uses separate tree–leaf coordinates, and its measurements belong to a different run.

A candidate is evaluated by its region. For a cell with membership mask $m$, the region's mean Brier loss is compared with that of the pooled remaining observations:

$$\Delta = \frac{1}{n_m}\sum_{i:\,m_i} L_i \;-\; \frac{1}{n-n_m}\sum_{i:\,\neg m_i} L_i, \qquad L_i=(y_i-p_i)^2 .$$

The region rule keeps cells with at least 50 observations and a share between 1% and 80%, then selects the eligible cell with the largest contrast $\Delta$. The Outer Array (Section 2.4) holds a single evaluation condition, the common 40,000-row discovery sample, so every candidate is scored on the same observations.

`read(RUN / "twinning_73101.json")` loads the search history for optimizer seed 73101 with a twinning initialization. The cell displays its first record, which is one of the initial designs, with its cell sizes and selected-region measurements.

In [2]:
run = read(RUN / "twinning_73101.json")
first = run["records"][0]
display(pd.DataFrame([{"medoids": first["medoids"], "selected_cells": first["region"]["cells"],
                       "outer_array": protocol["outer_array"][0]}]))
display(pd.DataFrame([{"cell_sizes": first["counts"], "region_n": first["region"]["n"],
                       "share": first["region"]["share"], "region_brier": first["region"]["loss"],
                       "rest_brier": first["region"]["rest_loss"], "contrast": first["region"]["contrast"]}]))
print("Predictor:", protocol["base_model"])
print("Auxiliary error model:", protocol["auxiliary"])
print("Eligible region share:", protocol["region_min_share"], "to", protocol["region_max_share"],
      "with at least", protocol["region_min_n"], "observations")
print("Search objectives:", protocol["objectives"])

,medoids,selected_cells,outer_array
0,"[95, 984, 2678, 2975]",[1],current rare-event population; same fixed 4000...


,cell_sizes,region_n,share,region_brier,rest_brier,contrast
0,"[12978, 8214, 6515, 12293]",8214,0.205350,0.054179,0.012384,0.041795


Predictor: unchanged saved logistic model
Auxiliary error model: unchanged saved depth-two 120-tree model
Eligible region share: 0.01 to 0.8 with at least 50 observations
Search objectives: ['largest eligible region Brier contrast versus pooled rest', 'selected region population share']


**Read the output.** The medoid entries are positions in the fixed 3,000-row medoid pool, and the four cell sizes sum to the 40,000 discovery observations. This initial candidate selects cell 1, which holds 8,214 observations (20.5% of the sample). Its mean Brier loss of 0.054179 exceeds the rest's 0.012384, giving a contrast of 0.041795. The contrast and the share are the two search objectives. They are outputs of evaluating the candidate on the Outer Array condition, whereas the medoids and the region rule are its inputs.

## Reconstruct the comparison

Search produces many candidates, and a larger contrast often comes with a smaller region. Ranking by contrast alone would favor very narrow regions, while ranking by share would favor broad ones. The Pareto comparison of Section 2.6 retains both kinds of candidate. Both objectives are maximized here, so a candidate is dominated when another is at least as large on contrast and share and strictly larger on one. The nondominated candidates form the empirical frontier.

`frontier(records)` orders valid records by decreasing contrast, with ties broken by decreasing share and then increasing ID. Walking down that order, it keeps a record only when its share exceeds that of every record kept before it. This produces the nondominated set and keeps the earliest ID when two records have identical objectives. The first assertion checks the result against the frontier stored with the search.

One candidate must then be chosen for confirmation. The declared primary policy restricts the frontier to regions with discovery share of at least 10% and takes the largest contrast, breaking ties by larger share and then earlier ID. The floor was fixed in the protocol before confirmation data existed, so it belongs to the procedure being validated. The second assertion checks that the policy reproduces the recorded primary selection.

In [3]:
kept = frontier(run["records"])
assert {r["id"] for r in kept} == set(run["frontier_ids"])
selected = max((r for r in kept if r["objectives"][1] >= .10),
               key=lambda r: (r["objectives"][0], r["objectives"][1], -r["id"]))
assert selected == run["primary"]
print("Candidates evaluated:", len(run["records"]), "Frontier size:", len(kept))
print("Primary policy:", protocol["primary_policy"])
display(pd.DataFrame([dict(candidate=r["id"], contrast=r["objectives"][0], share=r["objectives"][1],
                           eligible=r["objectives"][1] >= .10) for r in kept]))
print("Selected candidate:", selected["id"], "cells", selected["region"]["cells"],
      "contrast", round(selected["region"]["contrast"], 6), "share", selected["region"]["share"])

Candidates evaluated: 400 Frontier size: 39
Primary policy: maximum discovery contrast on Pareto frontier with share >= 0.10; tie larger share then earlier ID


,candidate,contrast,share,eligible
0,384,0.238311,0.010425,False
1,378,0.230937,0.010675,False
2,308,0.200790,0.010825,False
3,213,0.200212,0.010900,False
4,365,0.200071,0.010975,False
5,244,0.198873,0.011075,False
6,323,0.197944,0.011125,False
7,306,0.187947,0.011150,False
8,291,0.183409,0.012525,False
9,320,0.181077,0.014625,False


Selected candidate: 279 cells [0] contrast 0.061616 share 0.115625


**Read the output.** The seed-73101 history contains 400 evaluated candidates, of which 39 are nondominated. Along the frontier, contrast falls as share rises: candidate 384 reaches 0.238311 with 1.04% of the sample, while candidate 96 covers 62.3% with contrast 0.020030. No single ranking summarizes these tradeoffs. Among frontier regions with at least 10% share, candidate 279 has the largest contrast, 0.061616, with share 0.115625. Its discovery contrast was the maximum over the eligible candidates, so it is expected to be optimistic as an estimate of the region's contrast in new data, a selection effect analyzed in Chapter 9.

## Follow the frozen choice

A region selected for its discovery contrast needs a fresh assessment (Section 2.8). Before confirmation, the full rule is frozen: the logistic predictor, the auxiliary error model, the four medoid leaf-output vectors and the selected cell. A confirmation observation can then be assigned to the region from its features alone, before its outcome is consulted. `frozen["finalists"]` holds the frozen rule for every compared arm and seed, and `load()` has already checked its hash against the recorded results.

The first code block replays the frozen assignment. `assignments(choice)` routes the first 5,000 of the 200,000 confirmation rows through the saved auxiliary trees, measures Euclidean distance to the four stored medoid vectors and asserts that the resulting region membership equals the saved confirmation mask. The full sample is not replayed here to keep the cell short; the saved mask covers all 200,000 rows.

The second block recomputes the confirmation measurement from the saved observations. `confirmation()` returns the Brier loss of every confirmation row and the frozen region masks. `measure(loss, mask, z)` computes the regional and remaining Brier loss, their contrast and a standard error from the contrast's influence function,

$$\psi_i = \frac{m_i\,(L_i-\bar L_m)}{\pi} - \frac{(1-m_i)\,(L_i-\bar L_{\neg m})}{1-\pi}, \qquad \widehat{\mathrm{se}}(\hat\Delta)=\frac{\operatorname{sd}(\psi)}{\sqrt n},$$

where $\pi$ is the region's share. The study reports 19 claims, 13 confirmation contrasts and six paired differences, so each interval uses the Bonferroni critical value $z=\Phi^{-1}(1-0.05/(2\cdot 19))$. The resulting intervals hold simultaneously with approximate 95% coverage, conditional on the fitted models and frozen regions.

In [4]:
key = "twinning_73101"
choice = next(r for r in frozen["finalists"] if r["key"] == key)
assert choice["discovery"] == selected
labels, _ = assignments(choice)
print("Frozen cells:", choice["discovery"]["region"]["cells"],
      "Replayed rows:", len(labels), "Region members among them:", int(np.isin(labels, choice["discovery"]["region"]["cells"]).sum()))
from scipy.stats import norm
z = norm.ppf(1 - .05 / (2 * results["family_size"]))
assert np.isclose(z, results["critical_value"])
loss, masks = confirmation()
recomputed, _ = measure(loss, masks[key], z)
saved = results["confirmation"][key]
for field in ["n", "share", "brier", "rest_brier", "contrast", "standard_error"]:
    assert np.isclose(recomputed[field], saved[field], rtol=0, atol=1e-12), field
assert np.allclose(recomputed["simultaneous_ci"], saved["simultaneous_ci"], rtol=0, atol=1e-12)
print("Critical value:", z, "for", results["family_size"], "claims")
display(pd.DataFrame([{"split": "discovery", "share": selected["region"]["share"], "contrast": selected["region"]["contrast"]},
                      {"split": "confirmation", "share": saved["share"], "contrast": saved["contrast"]}]).set_index("split"))
display(pd.DataFrame([saved], index=[key]))

Frozen cells: [0] Replayed rows: 5000 Region members among them: 560
Critical value: 3.0077865564732638 for 19 claims


,share,contrast
split,,
discovery,0.115625,0.061616
confirmation,0.113925,0.057940


,n,share,brier,rest_brier,contrast,standard_error,simultaneous_ci
twinning_73101,22785,0.113925,0.072584,0.014645,0.057940,0.001525,"[0.05335135577493813, 0.06252789293682992]"


**Read the output.** The replay reproduces the saved membership for the first 5,000 confirmation rows, which confirms that the frozen rule assigns observations without their outcomes. On all 200,000 confirmation rows, the region holds 22,785 observations (11.39%). Its mean Brier loss is 0.072584 against 0.014645 for the rest, a contrast of 0.057940 with standard error 0.001525. The simultaneous interval [0.053351, 0.062528] excludes zero, so the elevated regional loss persists on new observations under the stated conditions. The confirmation contrast is below the discovery value of 0.061616, which is consistent with the selection optimism noted above. These are measurements of the frozen predictor's error in a synthetic population; they do not show that the predictor can be repaired or that the region has an operational interpretation.

## Check the claims

A validation report restates measured values, and each restated value should be traceable to the record that produced it. The evidence store built by `book/parallel_evidence/build.py` holds one record per measured quantity. Each record carries a scoped identifier, the value, the source file, that file's SHA-256 digest and a JSON pointer to the field inside the file.

`verify(store, claim)` takes a claim given as an identifier and a typed value. It recomputes the source file's digest, resolves the pointer and checks that the stored value still matches the source before comparing it with the claim. A claim is Supported when the values agree, within 0.5×10⁻⁶ for a continuous measurement so that six-decimal reporting is accepted. It is Contradicted when a record exists and the value differs, Absent when no record exists at the identifier and Unscorable when the claimed value has the wrong type or shape.

The cell checks three claims: the reported contrast rounded to six decimals, a different value at the same identifier and a claim of predictor repair for which the study recorded no measurement. The last table shows the evidence record that supports the first claim.

In [5]:
from book.parallel_evidence.build import OUT, verify
store = read(OUT / "store.json")
address = f"region:confirmation/{key}/contrast"
value = round(results["confirmation"][key]["contrast"], 6)
claims = [("reported contrast", address, value),
          ("altered contrast", address, 0.10),
          ("predictor repair", "region:experiment/predictor_repair_gain", 0.01)]
rows = [dict(claim=name, id=id_, value=claimed, verdict=verify(store, {"id": id_, "value": claimed})["verdict"])
        for name, id_, claimed in claims]
assert [r["verdict"] for r in rows] == ["Supported", "Contradicted", "Absent"]
display(pd.DataFrame(rows))
display(pd.DataFrame([verify(store, {"id": address, "value": value})["evidence"]]))

,claim,id,value,verdict
0,reported contrast,region:confirmation/twinning_73101/contrast,0.057940,Supported
1,altered contrast,region:confirmation/twinning_73101/contrast,0.100000,Contradicted
2,predictor repair,region:experiment/predictor_repair_gain,0.010000,Absent


,value,source,source_sha256,pointer
0,0.057940,book/evidence/aml_medoid_loop_leaf_output_2026...,fad0af92d04996ccb01f907412dc018c401f39d05443c6...,/confirmation/twinning_73101/contrast


**Read the output.** The rounded contrast 0.057940 is Supported by the record at `region:confirmation/twinning_73101/contrast`, whose source is the results file and pointer `/confirmation/twinning_73101/contrast`. The value 0.10 at the same identifier is Contradicted. The store has no record of a predictor-repair gain, so that claim is Absent; a Contradicted verdict would require a record holding a different value. The verifier compares supplied identifier–value pairs with records; extracting claims from free text and judging whether a sentence interprets a number correctly are separate problems, treated in Chapters 7 and 10.

## From a frozen test to a reported finding

The replay has followed one test through the complete ordinary loop. A medoid candidate defined a region, evaluation on the discovery sample produced its contrast and share, the Pareto comparison retained the tradeoffs among 400 candidates and a declared policy selected one for confirmation. The frozen rule then assigned new observations without their outcomes, the confirmation measurement was recomputed with a simultaneous interval and the reported value was bound to its source record.

Each stage answers a different question. The discovery contrast guided selection, the confirmation interval assesses the selected claim and the verifier checks that a reported number matches its record. None of these steps retrains the predictor or calls a language model. Chapter 3 examines the evaluator and the representation in which regions are formed; Chapter 4 develops the search that generated these 400 candidates.


## What to try

1. **Initial design and selection.** Compare the first record with candidate 279 and report how the cell sizes, selected share and discovery contrast changed. Then use the `operation` and `parent_id` fields to trace how candidate 279 was produced.
2. **The share floor.** Apply the primary policy with floors of 0.05 and 0.20 to the discovery frontier `kept`. Report the selected candidates and explain why the floor must be fixed before confirmation. Do not use `results["confirmation"]` to choose among floors.
3. **Other seeds.** Load `twinning_73102.json` and `twinning_73103.json`, recompute their frontiers and primary selections and compare them with their frozen finalists. The three seeds share one discovery sample, so their agreement is not an independent replication.
4. **Verifier outcomes.** Call `verify` with the discovery contrast `selected["region"]["contrast"]` at the confirmation identifier, and with the string `"0.057940"`. Explain why the first is Contradicted and the second Unscorable.
